In [16]:
import pandas as pd
import numpy as np
import re

In [ ]:
# Load the input table (microbial related citations until 2025)
df = pd.read_csv("../docs/extended/extended_data_table_3.tsv", sep="\t")
print(f"Total number of rows: {len(df)}")

Total number of rows: 3067


# Selecting a fixed random percentage from the microbial related citations

In [3]:
SEED = 42  # a fixed integer 
np.random.seed(SEED)

In [4]:
sample_fraction = 0.01  # 5% of the data
sample_size = int(len(df) * sample_fraction)

sampled_df = df.sample(
    n=sample_size,
    random_state=SEED
)

print(f"Selected {len(sampled_df)} rows ({sample_fraction*100:.0f}%)")

Selected 30 rows (1%)


In [ ]:
sampled_df = sampled_df.sort_index()
sampled_df.head()
sampled_df.to_csv( "extended_data_table_3_random_1_percent_automatic_curation_citations.tsv", sep="\t", index=False )

,Unnamed: 0,year,title,abstract,Targeted organisms,Technical target,Methods,Topics
102,106,2025,Clinical assessment and transcriptome analysis...,A glycoprotein-G-deleted live-attenuated vacci...,"Virus, Pathogen",Functional analysis,(Meta)transcriptomics,"Ecosystem Dynamics & Biodiversity, Health & Di..."
203,207,2024,Gene expression reprogramming of Pseudomonas a...,Abstract Different bacteria change their life ...,Bacteria,Functional analysis,(Meta)transcriptomics,Ecosystem Dynamics & Biodiversity
266,270,2023,Systematic SARS-CoV-2 S Gene Sequencing in Was...,"As the COVID-19 pandemic reached its peak, man...","Virus, Pathogen","Community (taxonomy) profiling, Variant",(Meta)genomics,"Ecosystem Dynamics & Biodiversity, Health & Di..."
485,490,2023,Comparative Microbiome Analysis of Three Epide...,(1) Background: Amplicon-based 16S rRNA profil...,"Bacteria, Microbiome","Community (taxonomy) profiling, Comparative an...",Metabarcoding,"Ecosystem Dynamics & Biodiversity, Health & Di..."
1089,1096,2025,Genomic Differences Between Two Fusarium oxysp...,The host specificity of Fusarium oxysporum (Fo...,Pathogen,NaN,(Meta)genomics,"Ecosystem Dynamics & Biodiversity, Health & Di..."


# Calculating stats between classifier and manual validation

In [ ]:
VALIDATION_PATH = "../docs/extended/extended_data_table_3_random_1_percent_automatic_curated_citations_manual_validation.tsv"
OUTPUT_ROW_LEVEL = "../docs/extended/extended_data_table_3_random_1_percent_automatic_curated_citations_manual_validation_row_level_keyword_confusion.tsv"
OUTPUT_SUMMARY = "../docs/extended/extended_data_table_3_random_1_percent_automatic_curated_citations_manual_validation_summary_keyword_confusion.tsv"

CATEGORIES = {
    "Targeted organisms": "Manual Targeted organisms",
    "Technical target": "Manual Technical target",
    "Methods": "Manual Methods",
    "Topics": "Manual Topics"
}

In [ ]:
# -----------------------------
# FIXED LABEL UNIVERSE (per category) FOR TN
# (All strings will be normalized to lowercase and collapsed spaces)
# -----------------------------
LABEL_UNIVERSE_RAW = {
    "Targeted organisms": [
        "Bacteria", "virus", "archaea", "eukaryot", "microbiome", "pathogen"
    ],
    "Technical target": [
        "Isolate",
        "communits (taxonomy) profiling",
        "functional analysis",
        "interactome",
        "amr",
        "MAGs",
        "gene identification/biomarker",
        "SNP",
        "(M)LST",
        "Annotation",
        "variant",
        "comparative analysis"
    ],
    "Methods": [
        "metabarcoding",
        "(meta)genomics",
        "metagenomics",
        "(meta)transcriptomics",
        "metatranscriptomics",
        "(meta)proteomics",
        "metabolomics",
        "imaging"
    ],
    "Topics": [
        "Ecosystem Dnamics & Biodiversity",
        "Health&Disease"
    ]
}

# -----------------------------
# CANONICALIZATION MAP
# -----------------------------
CANONICAL = {
    # Targeted organisms variants
    "eukaryot": "eukaryote",
    "eukaryote": "eukaryote",
    "eukaryotes": "eukaryote",

    # Technical target variants
    "communits (taxonomy) profiling": "community (taxonomy) profiling",
    "community (taxonomy) profiling": "community (taxonomy) profiling",
    "communities (taxonomy) profiling": "community (taxonomy) profiling",

    "(m)lst": "(m)lst",
    "(mlst)": "(m)lst",
    "mlst": "(m)lst",
    "mst": "(m)lst",

    "mags": "mags",
    "mag": "mags",

    "snp": "snp",
    "snps": "snp",

    # Topics variants
    "ecosystem dnamics & biodiversity": "ecosystem dynamics & biodiversity",
    "ecosystem dynamics & biodiversity": "ecosystem dynamics & biodiversity",

    "health&disease": "health&disease",
    "health & disease": "health&disease",
    "health &disease": "health&disease",
    "health and disease": "health&disease",
    "health & disease ": "health&disease",
}

def normalize_label(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return CANONICAL.get(s, s)

def cell_to_set(cell) -> set:
    """
    Convert a comma-separated string cell to a normalized set of keywords.
    - Empty/NaN -> empty set
    - lowercases, trims, collapses spaces
    - canonicalizes known variants/typos
    """
    if pd.isna(cell):
        return set()
    s = str(cell).strip()
    if not s:
        return set()

    parts = [p.strip() for p in s.split(",")]
    parts = [p for p in parts if p.strip()]
    return set(normalize_label(p) for p in parts)

def safe_div(num, den):
    return float(num) / float(den) if den else np.nan

# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_csv(VALIDATION_PATH, sep="\t")
print("Validation sample size:", len(df))

missing_cols = [c for c in list(CATEGORIES.keys()) + list(CATEGORIES.values()) if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing expected columns in validation file: {missing_cols}")

# -----------------------------
# BUILD FIXED LABEL UNIVERSE (normalized)
# -----------------------------
label_universe = {}
for cat, labels in LABEL_UNIVERSE_RAW.items():
    label_universe[cat] = set(normalize_label(x) for x in labels)

# Quick sanity check: every category in CATEGORIES should have a universe
for cat in CATEGORIES.keys():
    if cat not in label_universe:
        raise ValueError(f"Missing label universe definition for category: {cat}")
    print(f"Universe size for '{cat}': {len(label_universe[cat])}")

# -----------------------------
# ROW-LEVEL CONFUSION (keyword-level) + AUDIT LABELS
# -----------------------------
row_level_records = []

# Keep some ID columns if present
possible_id_cols = ["year", "title", "doi", "DOI", "PublicationID", "pub_id", "pmid"]
id_cols = [c for c in possible_id_cols if c in df.columns]

for idx, row in df.iterrows():
    out = {"row_index": idx}
    for c in id_cols:
        out[c] = row.get(c)

    for auto_col, manual_col in CATEGORIES.items():
        A = cell_to_set(row[auto_col])       # automatic keywords
        M = cell_to_set(row[manual_col])     # manual keywords
        U = label_universe[auto_col]         # fixed universe keywords

        tp_set = A & M
        fp_set = A - M
        fn_set = M - A
        tn_set = U - (A | M)

        out[f"{auto_col} TP"] = len(tp_set)
        out[f"{auto_col} FP"] = len(fp_set)
        out[f"{auto_col} FN"] = len(fn_set)
        out[f"{auto_col} TN"] = len(tn_set)

        # Store labels for inspection / audit trail
        out[f"{auto_col} TP_labels"] = ", ".join(sorted(tp_set))
        out[f"{auto_col} FP_labels"] = ", ".join(sorted(fp_set))
        out[f"{auto_col} FN_labels"] = ", ".join(sorted(fn_set))

    row_level_records.append(out)

row_level_df = pd.DataFrame(row_level_records)
print("Row-level table created:", row_level_df.shape)

row_level_df.to_csv(OUTPUT_ROW_LEVEL, sep="\t", index=False)
print("Saved:", OUTPUT_ROW_LEVEL)

# -----------------------------
# CATEGORY-LEVEL SUMMARY (aggregate TP/FP/FN/TN) + P/R/F1
# -----------------------------
summary_records = []

for auto_col, manual_col in CATEGORIES.items():
    TP = int(row_level_df[f"{auto_col} TP"].sum())
    FP = int(row_level_df[f"{auto_col} FP"].sum())
    FN = int(row_level_df[f"{auto_col} FN"].sum())
    TN = int(row_level_df[f"{auto_col} TN"].sum())

    precision = safe_div(TP, TP + FP)
    recall = safe_div(TP, TP + FN)
    f1 = safe_div(2 * precision * recall, precision + recall)

    summary_records.append({
        "Category": auto_col,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Precision_%": None if np.isnan(precision) else round(precision * 100, 1),
        "Recall_%": None if np.isnan(recall) else round(recall * 100, 1),
        "F1_%": None if np.isnan(f1) else round(f1 * 100, 1),
        "Universe_size": len(label_universe[auto_col]),
    })

summary_df = pd.DataFrame(summary_records)

print("\nCategory-level summary:")
print(summary_df.to_string(index=False))

summary_df.to_csv(OUTPUT_SUMMARY, sep="\t", index=False)
print("Saved:", OUTPUT_SUMMARY)

# -----------------------------
# MICRO + MACRO AVERAGES
# -----------------------------
macro_precision = summary_df["Precision"].mean()
macro_recall = summary_df["Recall"].mean()
macro_f1 = summary_df["F1"].mean()

total_TP = int(summary_df["TP"].sum())
total_FP = int(summary_df["FP"].sum())
total_FN = int(summary_df["FN"].sum())

micro_precision = safe_div(total_TP, total_TP + total_FP)
micro_recall = safe_div(total_TP, total_TP + total_FN)
micro_f1 = safe_div(2 * micro_precision * micro_recall, micro_precision + micro_recall)

print("\nAverages:")
print(f"Macro Precision: {macro_precision:.3f} ({macro_precision*100:.1f}%)")
print(f"Macro Recall:    {macro_recall:.3f} ({macro_recall*100:.1f}%)")
print(f"Macro F1:        {macro_f1:.3f} ({macro_f1*100:.1f}%)")
print(f"Micro Precision: {micro_precision:.3f} ({micro_precision*100:.1f}%)")
print(f"Micro Recall:    {micro_recall:.3f} ({micro_recall*100:.1f}%)")
print(f"Micro F1:        {micro_f1:.3f} ({micro_f1*100:.1f}%)")

# -----------------------------
# Summarised stats for the paper
# -----------------------------
n_publications = len(df)

f1_min = float(summary_df["F1_%"].min())
f1_max = float(summary_df["F1_%"].max())
p_min = float(summary_df["Precision_%"].min())
p_max = float(summary_df["Precision_%"].max())
r_min = float(summary_df["Recall_%"].min())
r_max = float(summary_df["Recall_%"].max())


Validation sample size: 30
Universe size for 'Targeted organisms': 6
Universe size for 'Technical target': 12
Universe size for 'Methods': 8
Universe size for 'Topics': 2
Row-level table created: (30, 31)
Saved: ../docs/extended/manual_validation_row_level_keyword_confusion.tsv

Category-level summary:
          Category  TP  FP  FN  TN  Precision   Recall       F1  Precision_%  Recall_%  F1_%  Universe_size
Targeted organisms  34  11  20 115   0.755556 0.629630 0.686869         75.6      63.0  68.7              6
  Technical target  22  14  72 268   0.611111 0.234043 0.338462         61.1      23.4  33.8             12
           Methods  27   2   8 203   0.931034 0.771429 0.843750         93.1      77.1  84.4              8
            Topics  35   5   4  16   0.875000 0.897436 0.886076         87.5      89.7  88.6              2
Saved: ../docs/extended/manual_validation_summary_keyword_confusion.tsv

Averages:
Macro Precision: 0.793 (79.3%)
Macro Recall:    0.633 (63.3%)
Macro F1:  